# Día 19 — Limpieza de datos y Joins en pandas

**Dataset:** Tienda Tech — `productos.csv` + `ventas.csv`  
**Objetivo:** detectar y tratar problemas en datos reales antes de analizarlos.

---

In [1]:
import pandas as pd

productos = pd.read_csv('../data/day19/productos.csv')
ventas    = pd.read_csv('../data/day19/ventas.csv')

print('Productos:', productos.shape)
print('Ventas:   ', ventas.shape)

Productos: (50, 6)
Ventas:    (50, 6)


---
## Concepto 1 — Detectar nulos: `df.isnull().sum()`

Los datos reales siempre tienen valores vacíos. Antes de analizar hay que saber exactamente dónde están y cuántos.

- `df.isnull()` devuelve un DataFrame de booleanos: `True` donde hay nulo
- `.sum()` suma los `True` por columna (True = 1, False = 0)
- Dividir entre `len(df)` da el porcentaje de nulos

**Ejercicio 1:** detectar nulos en `productos` y mostrar su porcentaje por columna.

In [2]:
nulos = productos.isnull().sum()
pct   = (nulos / len(productos) * 100).round(2)

resumen = pd.DataFrame({'nulos': nulos, 'porcentaje_%': pct})
print(resumen[resumen['nulos'] > 0])

           nulos  porcentaje_%
categoria      1           2.0
marca          1           2.0
precio         2           4.0
stock          4           8.0


---
## Concepto 2 — Rellenar o eliminar nulos: `fillna()` vs `dropna()`

No todos los nulos se tratan igual. La decisión depende de lo que representa la columna:

- `df['col'].fillna(valor)` → sustituye el nulo por un valor concreto
- `df.dropna(subset=['col'])` → elimina filas donde esa columna es nula

**Regla:** usa `fillna` cuando el nulo tiene un significado lógico (sin stock = 0).  
Usa `dropna` cuando la fila no puede analizarse sin ese dato (sin precio = no analizable).

**Ejercicio 2:** en `productos`, rellenar `stock` nulo con 0 y eliminar filas donde `precio` es nulo.

In [3]:
# stock nulo = sin existencias → rellenar con 0
productos['stock'] = productos['stock'].fillna(0)

# precio nulo = producto no analizable → eliminar fila
filas_antes = len(productos)
productos = productos.dropna(subset=['precio'])

print('Filas eliminadas por precio nulo:', filas_antes - len(productos))
print('Nulos restantes en stock y precio:')
print(productos[['stock', 'precio']].isnull().sum())

Filas eliminadas por precio nulo: 2
Nulos restantes en stock y precio:
stock     0
precio    0
dtype: int64


---
## Concepto 3 — Eliminar duplicados: `drop_duplicates()`

Una fila duplicada es idéntica a otra que ya existe en el DataFrame. Sin eliminarlas, las sumas y conteos estarán inflados.

- `df.duplicated()` → Serie booleana: `True` donde la fila es un duplicado
- `df.duplicated(keep=False)` → marca **todas** las ocurrencias, no solo la segunda
- `df.drop_duplicates()` → elimina duplicados y conserva la primera ocurrencia

**Ejercicio 3:** detectar y eliminar filas duplicadas en `productos`.

In [4]:
print('Filas duplicadas:', productos.duplicated().sum())

print('Detalle de duplicados:')
print(productos[productos.duplicated(keep=False)][['producto_id', 'nombre']].sort_values('producto_id'))

productos = productos.drop_duplicates()
print('Shape final:', productos.shape)

Filas duplicadas: 3
Detalle de duplicados:
    producto_id                nombre
0           101    Laptop ProBook 450
47          101    Laptop ProBook 450
1           102  Monitor UltraWide 34
48          102  Monitor UltraWide 34
2           103   Teclado Mecánico K3
49          103   Teclado Mecánico K3
Shape final: (45, 6)


---
## Concepto 4 — Verificar tipos: `df.dtypes`

Una columna con números guardados como texto (`object`) no se puede sumar ni calcular. Hay que convertirla primero.

- `df.dtypes` → muestra el tipo de cada columna
- `pd.to_datetime(df['col'], format='%d/%m/%Y')` → convierte string a fecha
- `df['col'].str.replace('€', '').astype(float)` → quita el símbolo y convierte a número

**Ejercicio 4:** corregir `fecha` (string `dd/mm/yyyy`) y `total` (string con `€`) en `ventas`.

In [5]:
print('Tipos actuales en ventas:')
print(ventas.dtypes)

# fecha: 'dd/mm/yyyy' → datetime
ventas['fecha'] = pd.to_datetime(ventas['fecha'], format='%d/%m/%Y')

# total: '1234.50€' → float
ventas['total'] = ventas['total'].str.replace('€', '', regex=False).astype(float)

print('Tipos corregidos:')
print(ventas[['fecha', 'total']].dtypes)
print(ventas[['fecha', 'total']].head(3))

Tipos actuales en ventas:
venta_id       int64
producto_id    int64
fecha            str
cantidad       int64
total            str
cliente          str
dtype: object
Tipos corregidos:
fecha    datetime64[us]
total           float64
dtype: object
       fecha    total
0 2024-01-15  1799.98
1 2024-01-22   129.99
2 2024-01-30   839.97


---
## Concepto 5 — JOIN en pandas: `pd.merge()`

`pd.merge()` une dos DataFrames a través de una columna común. Es el equivalente al `INNER JOIN` de SQL.

```
pd.merge(df_izquierdo, df_derecho, on='columna_comun', how='inner')
```

- `on=` → columna con el mismo nombre en ambos DataFrames
- `how='inner'` → solo filas con coincidencia en ambos lados (es el valor por defecto)

**Ejercicio 5:** unir `ventas` con `productos` usando `producto_id`.

In [6]:
ventas_detalle = pd.merge(ventas, productos, on='producto_id', how='inner')

print('Ventas originales:   ', len(ventas))
print('Resultado del merge: ', len(ventas_detalle))
print('Primeras filas:')
print(ventas_detalle[['venta_id', 'producto_id', 'nombre', 'categoria', 'total']].head())

Ventas originales:    50
Resultado del merge:  47
Primeras filas:
   venta_id  producto_id                   nombre    categoria    total
0         1          101       Laptop ProBook 450      Laptops  1799.98
1         2          103      Teclado Mecánico K3     Teclados   129.99
2         3          115  Auriculares AirPods Pro  Auriculares   839.97
3         4          107            Monitor 4K 27    Monitores   399.99
4         5          102     Monitor UltraWide 34    Monitores   899.98


---
## Concepto 6 — Tipos de merge: parámetro `how=`

| how | Qué devuelve | Equivalente SQL |
|-----|-------------|------------------|
| `inner` | Solo filas con coincidencia en ambos lados | INNER JOIN |
| `left` | Todas las filas del izquierdo, NaN si no hay coincidencia | LEFT JOIN |
| `right` | Todas las filas del derecho, NaN si no hay coincidencia | RIGHT JOIN |
| `outer` | Todas las filas de ambos lados | FULL OUTER JOIN |

El más común en análisis es `left`: conserva todos los registros del DataFrame principal aunque no tengan coincidencia en el otro.

**Ejercicio 6:** comparar cuántas filas devuelve `inner` vs `left` entre `ventas` y `productos`.

In [7]:
merge_inner = pd.merge(ventas, productos, on='producto_id', how='inner')
merge_left  = pd.merge(ventas, productos, on='producto_id', how='left')

print('Inner:', len(merge_inner), 'filas')
print('Left: ', len(merge_left),  'filas')
print('Ventas sin producto en catálogo:', len(merge_left) - len(merge_inner))

huerfanas = merge_left[merge_left['nombre'].isnull()][['venta_id', 'producto_id']]
print('Ventas sin producto registrado:')
print(huerfanas)

Inner: 47 filas
Left:  50 filas
Ventas sin producto en catálogo: 3
Ventas sin producto registrado:
    venta_id  producto_id
47        48          148
48        49          149
49        50          150


---
## Concepto 7 — Verificar el resultado del merge

Después de un merge siempre hay que responder tres preguntas:

1. **¿Se perdieron filas?** Si el resultado tiene menos filas que el DataFrame izquierdo, hay claves sin coincidencia
2. **¿Se duplicaron filas?** Si tiene más filas, hay claves repetidas en el DataFrame derecho
3. **¿La clave sigue siendo única?** Verificar con `df['id'].duplicated().sum()`

Un merge 1:1 correcto devuelve exactamente las mismas filas que el DataFrame izquierdo.

**Ejercicio 7:** verificar la integridad del merge entre `ventas` y `productos`.

In [8]:
print('Ventas antes del merge: ', len(ventas))
print('Ventas después (inner): ', len(ventas_detalle))

diff = len(ventas_detalle) - len(ventas)
if diff > 0:
    print('ALERTA: el merge añadió', diff, 'filas → hay duplicados en productos')
elif diff < 0:
    print('INFO: se perdieron', abs(diff), 'ventas → no tienen producto registrado')
else:
    print('OK: merge 1:1, sin pérdidas ni duplicaciones')

duplicados_id = ventas_detalle['venta_id'].duplicated().sum()
print('venta_ids duplicados post-merge:', duplicados_id)

Ventas antes del merge:  50
Ventas después (inner):  47
INFO: se perdieron 3 ventas → no tienen producto registrado
venta_ids duplicados post-merge: 0
